In [ ]:
%pip install -q openeo geopandas rasterio

In [ ]:
import json
import openeo

# Akan menampilkan URL + kode (device flow). Buka di browser mana saja, login CDSE, lalu kembali ke sini.
conn = openeo.connect('openeo.dataspace.copernicus.eu').authenticate_oidc()

In [ ]:
with open('aoi.geojson') as f:
    aoi = json.load(f)

BANDS = ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12']

cube = conn.load_collection(
    'SENTINEL2_L2A',
    temporal_extent=['2026-06-01', '2026-09-30'],
    bands=BANDS + ['SCL'],
    max_cloud_cover=40,
)
cube = cube.filter_spatial(aoi)

# Buang awan, bayangan awan, cirrus, salju, saturated, no data (berdasarkan SCL)
scl = cube.band('SCL')
bad = (scl == 0) | (scl == 1) | (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10) | (scl == 11)
cube = cube.mask(bad).filter_bands(BANDS)

# Komposit median per piksel, samakan resolusi 20 m dan CRS UTM 49S
cube = cube.reduce_dimension(dimension='t', reducer='median')
cube = cube.resample_spatial(resolution=20, projection=32749)
cube = cube.save_result(format='GTiff')

In [ ]:
# AoI sebesar ini harus lewat batch job (bukan download langsung)
job = cube.create_job(title='s2_aoi_median_20m')
job.start_and_wait()

In [ ]:
job.get_results().download_files('openeo_out')

In [ ]:
import glob
import rasterio

for p in glob.glob('openeo_out/*.tif*'):
    with rasterio.open(p) as src:
        print(p, src.count, 'band,', src.width, 'x', src.height, src.crs)